In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Load the two main files we need
# Replace 'path/to/' with wherever you saved the ml-latest-small folder
movies = pd.read_csv('../data/ml-latest-small/movies.csv')
ratings = pd.read_csv('../data/ml-latest-small/ratings.csv')

# Let's see what's inside
print("MOVIES DATA:")
print(movies.head())
print(f"\nShape: {movies.shape}")
print(f"\nColumns: {movies.columns.tolist()}")
print ("\ninfo: \n")
movies.info()

print("\n" + "="*50 + "\n")

print("RATINGS DATA:")
print(ratings.head())
print(f"\nShape: {ratings.shape}")
print(f"\nColumns: {ratings.columns.tolist()}")
print("\ninfo: \n")
ratings.info()

MOVIES DATA:
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  

Shape: (9742, 3)

Columns: ['movieId', 'title', 'genres']

info: 

<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
dtypes: int64(1), str(2)
memory usage: 228.5 KB



In [3]:
# number of unique users:
users = ratings['userId']
print("number of unique users: ", len(users.unique()))
print()

# range of rating:
ratingCol = ratings['rating']
print("min rating: ", ratingCol.min())
print("max rating: ", ratingCol.max())

# most common rating
ratingFreq = ratingCol.value_counts()
print (ratingFreq)

number of unique users:  610

min rating:  0.5
max rating:  5.0
rating
4.0    26818
3.0    20047
5.0    13211
3.5    13136
4.5     8551
2.0     7551
2.5     5550
1.0     2811
1.5     1791
0.5     1370
Name: count, dtype: int64


In [4]:
# merging both dataframes:
data = pd.merge(ratings, movies, how="inner", on="movieId")
print (data.head())

   userId  movieId  rating  timestamp                        title  \
0       1        1     4.0  964982703             Toy Story (1995)   
1       1        3     4.0  964981247      Grumpier Old Men (1995)   
2       1        6     4.0  964982224                  Heat (1995)   
3       1       47     5.0  964983815  Seven (a.k.a. Se7en) (1995)   
4       1       50     5.0  964982931   Usual Suspects, The (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                               Comedy|Romance  
2                        Action|Crime|Thriller  
3                             Mystery|Thriller  
4                       Crime|Mystery|Thriller  


In [5]:
# user_movie_matrix = data.pivot_table(values="rating", index='userId', columns="title")
# print(user_movie_matrix.shape)

In [6]:
# movie_similarity = user_movie_matrix.corr()
# print (movie_similarity.shape)

# too big data and not a cleaned one so we will filter out first

In [7]:
rating_counts = data.groupby("title")["rating"].size()
print (rating_counts)
print (len(rating_counts))

title
'71 (2014)                                    1
'Hellboy': The Seeds of Creation (2004)       1
'Round Midnight (1986)                        2
'Salem's Lot (2004)                           1
'Til There Was You (1997)                     2
                                             ..
eXistenZ (1999)                              22
xXx (2002)                                   24
xXx: State of the Union (2005)                5
¡Three Amigos! (1986)                        26
À nous la liberté (Freedom for Us) (1931)     1
Name: rating, Length: 9719, dtype: int64
9719


In [8]:
popular_movies = rating_counts[rating_counts >= 50]
print (len(popular_movies))
print (popular_movies.head(10))

450
title
10 Things I Hate About You (1999)         54
12 Angry Men (1957)                       57
2001: A Space Odyssey (1968)             109
28 Days Later (2002)                      58
300 (2007)                                80
40-Year-Old Virgin, The (2005)            74
A.I. Artificial Intelligence (2001)       56
Abyss, The (1989)                         62
Ace Ventura: Pet Detective (1994)        161
Ace Ventura: When Nature Calls (1995)     88
Name: rating, dtype: int64


In [9]:
data_popular = data[data['title'].isin(popular_movies.index)]
print(data_popular.shape)

(41362, 6)


In [10]:
user_movie_matrix = data_popular.pivot_table(values="rating", index="userId", columns="title")
print(user_movie_matrix.shape)
user_movie_matrix_filled = user_movie_matrix.fillna(0)

(606, 450)


In [11]:
movie_similarity = cosine_similarity(user_movie_matrix_filled.T)
movie_similarity = pd.DataFrame(movie_similarity, index=user_movie_matrix.columns, columns=user_movie_matrix.columns)


In [12]:
def get_recommendation(movie_title):
    similarity_scores = movie_similarity[movie_title]
    ordered_scores = similarity_scores.sort_values(ascending=False)
    return ordered_scores.iloc[1:6]

In [13]:
print(get_recommendation("Forrest Gump (1994)"))
print()
print()
print(get_recommendation("Toy Story (1995)"))
print()
print()
print(get_recommendation("Matrix, The (1999)"))

title
Shawshank Redemption, The (1994)    0.712993
Jurassic Park (1993)                0.688259
Pulp Fiction (1994)                 0.685544
Braveheart (1995)                   0.643090
Silence of the Lambs, The (1991)    0.639463
Name: Forrest Gump (1994), dtype: float64


title
Toy Story 2 (1999)                           0.572601
Jurassic Park (1993)                         0.565637
Independence Day (a.k.a. ID4) (1996)         0.564262
Star Wars: Episode IV - A New Hope (1977)    0.557388
Forrest Gump (1994)                          0.547096
Name: Toy Story (1995), dtype: float64


title
Fight Club (1999)                                        0.713937
Star Wars: Episode V - The Empire Strikes Back (1980)    0.700935
Saving Private Ryan (1998)                               0.679615
Star Wars: Episode IV - A New Hope (1977)                0.663447
Star Wars: Episode VI - Return of the Jedi (1983)        0.660984
Name: Matrix, The (1999), dtype: float64


In [14]:
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split

In [15]:
reader = Reader(rating_scale=(0.5, 5.0))
data_surprise = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)


In [16]:
trainset, testset = train_test_split(data_surprise, test_size=0.2)
model = SVD()
model.fit(trainset)

In [17]:
from surprise import accuracy

predictions = model.test(testset)
rmse = accuracy.rmse(predictions)

RMSE: 0.8791


In [18]:
# Test: Predict what user 1 would rate a specific movie
prediction = model.predict(uid=1, iid=50)
print(f"Predicted rating for user 1, movie 50: {prediction.est:.2f}")

Predicted rating for user 1, movie 50: 5.00


In [19]:
# Create title to movieId mapping
title_to_id = dict(zip(movies['title'], movies['movieId']))

# Test it
print(title_to_id['Matrix, The (1999)'])

2571


In [32]:
def recommend_with_retrain(favorite_movies_dict, n=10):
    """
    Retrain model with user's favorites included
    """
    # Step 1: Create new user's ratings
    new_user_id = 999
    new_ratings = []
    
    for title, rating in favorite_movies_dict.items():
        movie_id = title_to_id[title]
        new_ratings.append({
            'userId': new_user_id,
            'movieId': movie_id,
            'rating': rating
        })
    
    # Step 2: Add to original ratings
    new_ratings_df = pd.DataFrame(new_ratings)
    combined_ratings = pd.concat([ratings, new_ratings_df], ignore_index=True)
    
    # Step 3: Prepare data for Surprise
    data_with_new_user = Dataset.load_from_df(
        combined_ratings[['userId', 'movieId', 'rating']], 
        reader
    )
    
    # Step 4: Retrain on full dataset (no train/test split this time)
    trainset_full = data_with_new_user.build_full_trainset()
    
    new_model = SVD()
    new_model.fit(trainset_full)
    
    # Step 5: Get all movies user hasn't rated
    favorite_ids = [title_to_id[title] for title in favorite_movies_dict.keys()]
    all_movie_ids = movies['movieId'].tolist()
    unwatched_ids = [mid for mid in all_movie_ids if mid not in favorite_ids]
    
    # Step 6: Predict for unwatched movies
    predictions = []
    for movie_id in unwatched_ids:
        pred = new_model.predict(uid=new_user_id, iid=movie_id)
        predictions.append((movie_id, pred.est))
    
    # Step 7: Sort and return top N
    predictions.sort(key=lambda x: x[1], reverse=True)
    
    result = []
    for mid, rating in predictions[:n]:
        title = movies[movies['movieId'] == mid]['title'].values[0]
        result.append({'title': title, 'predicted_rating': round(rating, 2)})
    
    return result

In [35]:
action_favorites = {
    'Die Hard (1988)': 5.0,
    'Terminator 2: Judgment Day (1991)': 5.0,
    'Speed (1994)': 5.0
}

print("\n\n🎬 RECOMMENDATIONS WITH RETRAINING (Action taste):\n")
recommendations = recommend_with_retrain(action_favorites, n=10)
for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec['title']} - {rec['predicted_rating']}★")



🎬 RECOMMENDATIONS WITH RETRAINING (Action taste):

1. Dark Knight, The (2008) - 4.75★
2. Goodfellas (1990) - 4.73★
3. Rear Window (1954) - 4.73★
4. Full Metal Jacket (1987) - 4.7★
5. Star Wars: Episode V - The Empire Strikes Back (1980) - 4.7★
6. Blade Runner (1982) - 4.68★
7. Shawshank Redemption, The (1994) - 4.66★
8. Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001) - 4.66★
9. Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981) - 4.66★
10. Princess Bride, The (1987) - 4.64★


In [29]:
print("\n📊 COSINE SIMILARITY RECOMMENDATIONS:\n")
print("Based on Matrix:")
print(get_recommendation("Matrix, The (1999)"))


📊 COSINE SIMILARITY RECOMMENDATIONS:

Based on Matrix:
title
Fight Club (1999)                                        0.713937
Star Wars: Episode V - The Empire Strikes Back (1980)    0.700935
Saving Private Ryan (1998)                               0.679615
Star Wars: Episode IV - A New Hope (1977)                0.663447
Star Wars: Episode VI - Return of the Jedi (1983)        0.660984
Name: Matrix, The (1999), dtype: float64


In [36]:
def get_complete_recommendations(favorite_movies_dict):
    """
    Show both recommendation types side by side
    """
    print("🎬 MOVIE RECOMMENDATIONS FOR YOU\n")
    print(f"📊 Based on your favorites: {', '.join(favorite_movies_dict.keys())}\n")
    print("━" * 80)
    
    # Part 1: ML Model Predictions
    print("\n🎯 PERSONALIZED PICKS (ML Model - SVD)")
    print("Trained on 100K ratings to predict YOUR taste\n")
    
    ml_recs = recommend_with_retrain(favorite_movies_dict, n=5)
    for i, rec in enumerate(ml_recs, 1):
        print(f"{i}. {rec['title']} - Predicted Rating: {rec['predicted_rating']}★")
    
    print("\n" + "━" * 80)
    
    # Part 2: Cosine Similarity
    print("\n💡 SIMILAR TO YOUR FAVORITES")
    print("Movies with similar rating patterns\n")
    
    # Get recommendations for each favorite and combine
    all_similar = {}
    for movie_title in favorite_movies_dict.keys():
        similar = get_recommendation(movie_title)
        for title, score in similar.items():
            if title not in all_similar:
                all_similar[title] = score
            else:
                all_similar[title] = max(all_similar[title], score)  # Keep highest score
    
    # Sort and get top 5
    sorted_similar = sorted(all_similar.items(), key=lambda x: x[1], reverse=True)[:5]
    
    for i, (title, score) in enumerate(sorted_similar, 1):
        print(f"{i}. {title} - Similarity: {score:.2f}")
    
    print("\n" + "━" * 80)

In [37]:
action_favorites = {
    'Die Hard (1988)': 5.0,
    'Terminator 2: Judgment Day (1991)': 5.0,
    'Speed (1994)': 5.0
}

get_complete_recommendations(action_favorites)

🎬 MOVIE RECOMMENDATIONS FOR YOU

📊 Based on your favorites: Die Hard (1988), Terminator 2: Judgment Day (1991), Speed (1994)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🎯 PERSONALIZED PICKS (ML Model - SVD)
Trained on 100K ratings to predict YOUR taste

1. Matrix, The (1999) - Predicted Rating: 4.85★
2. Good, the Bad and the Ugly, The (Buono, il brutto, il cattivo, Il) (1966) - Predicted Rating: 4.75★
3. Star Wars: Episode V - The Empire Strikes Back (1980) - Predicted Rating: 4.74★
4. Miller's Crossing (1990) - Predicted Rating: 4.72★
5. Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981) - Predicted Rating: 4.71★

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

💡 SIMILAR TO YOUR FAVORITES
Movies with similar rating patterns

1. Jurassic Park (1993) - Similarity: 0.72
2. Terminator, The (1984) - Similarity: 0.70
3. True Lies (1994) - Similarity: 0.67
4. Indiana Jones and the Last Crusad